# TRM-derived feature model: Ridge regression

This experiment evaluates a first feature-based model against persistence and the existing ARIMA level forecasts. The primary feature set is deliberately limited to TRM-derived variables because this repository contains no external macroeconomic data with historical publication/availability timestamps.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
from trm_forecast.business.evaluation import evaluate_business_value, validate_forecast_grid
from trm_forecast.evaluation.arima import evaluate_arima
from trm_forecast.evaluation.persistence import evaluate_persistence
from trm_forecast.evaluation.ridge import evaluate_ridge
from trm_forecast.evaluation.protocol import PartitionSpec, TemporalEvaluationProtocol

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data/interim/trm_normalized.csv').exists())
config = yaml.safe_load((ROOT / 'configs/base.yaml').read_text())
evaluation = config['evaluation']
data = pd.read_csv(ROOT / 'data/interim/trm_normalized.csv', parse_dates=['date'])
series = data.set_index('date')['trm'].sort_index()
specs = tuple(PartitionSpec(name, values['start'], values['end']) for name, values in evaluation['partitions'].items())
result = TemporalEvaluationProtocol(evaluation['as_of_date'], tuple(evaluation['horizons']), specs).split(series)
horizons = tuple(evaluation['horizons'])
usd_amount = 100_000
ridge_kwargs = dict(lags=tuple(config['features']['lags']), rolling_windows=tuple(config['features']['rolling_windows']))
arima_kwargs = dict(min_history=config['arima']['min_history'], maxiter=config['arima']['maxiter'], refit_frequency=config['arima']['refit_frequency'])

## Feature definition and availability

Features are computed at the forecast origin: the observed origin TRM level, lagged TRM levels, the log return through the current origin, and rolling mean/standard deviation of `trm.shift(1)`. No external variables are used. The repository contains no macroeconomic dataset or historical release-vintage field from which publication-time joins could be reconstructed, so US rates, dollar indices, Colombian policy rates, and inflation are excluded from the primary leakage-safe experiment.

In [ ]:
# Select Ridge alpha on validation only. The test partition is not used here.
alpha_rows = []
for alpha in config['ridge']['alpha_grid']:
    _, metrics = evaluate_ridge(result, alpha=float(alpha), horizons=horizons, partitions=('validation',), **ridge_kwargs)
    alpha_rows.append({'alpha': float(alpha), 'mean_validation_mae': metrics['mae'].mean(), 'mean_validation_rmse': metrics['rmse'].mean(), 'failed_predictions': int(metrics['failed_predictions'].sum())})
alpha_selection = pd.DataFrame(alpha_rows).sort_values(['mean_validation_mae', 'alpha'])
selected_alpha = float(alpha_selection.iloc[0]['alpha'])
alpha_selection

In [ ]:
# Evaluate the frozen Ridge alpha and existing level models on validation/test.
ridge_validation, ridge_validation_metrics = evaluate_ridge(result, selected_alpha, horizons, ('validation',), **ridge_kwargs)
ridge_test, ridge_test_metrics = evaluate_ridge(result, selected_alpha, horizons, ('test',), **ridge_kwargs)
persistence, persistence_metrics = evaluate_persistence(result, horizons)
persistence = persistence[persistence['partition'].isin(['validation', 'test'])].copy()
arima010, arima010_metrics = evaluate_arima(result, (0, 1, 0), horizons, ('validation', 'test'), **arima_kwargs)
arima011, arima011_metrics = evaluate_arima(result, (0, 1, 1), horizons, ('validation', 'test'), **arima_kwargs)
arima010_drift, arima010_drift_metrics = evaluate_arima(result, (0, 1, 0), horizons, ('validation', 'test'), trend=config['arima']['drift_trend'], **arima_kwargs)

def successful(frame):
    return frame[frame['status'].isin(['converged', 'converged_with_warning'])].copy() if 'status' in frame else frame.copy()

forecasts = pd.concat([persistence, successful(ridge_validation), successful(ridge_test), successful(arima010), successful(arima011), successful(arima010_drift)], ignore_index=True)
forecasts['model'] = forecasts['model'].replace({'arima(0, 1, 0)': 'arima_010', 'arima(0, 1, 1)': 'arima_011', 'arima(0, 1, 0)_drift': 'arima_010_drift'})
forecast_metrics = pd.concat([
    persistence_metrics.assign(model='persistence_last', failed_predictions=0),
    ridge_validation_metrics.assign(model='ridge_trm'), ridge_test_metrics.assign(model='ridge_trm'),
    arima010_metrics.assign(model='arima_010'), arima011_metrics.assign(model='arima_011'),
    arima010_drift_metrics.assign(model='arima_010_drift'),
], ignore_index=True)
forecast_metrics[forecast_metrics['partition'].isin(['validation', 'test'])][['partition', 'model', 'horizon', 'observations', 'failed_predictions', 'mae', 'rmse']].sort_values(['partition', 'horizon', 'model'])

In [ ]:
# Keep only complete origin/model groups common to every compared model.
issues = validate_forecast_grid(forecasts, horizons, strict=False)
if issues.empty:
    exclusions = pd.DataFrame(columns=['partition', 'model', 'issue', 'excluded_origin_model_groups', 'excluded_operation_decisions'])
else:
    issues['partition'] = issues['_partition']
    exclusions = (issues.drop_duplicates(['_partition', 'forecast_origin', 'model', 'issue'])
        .groupby(['partition', 'model', 'issue'], as_index=False).size()
        .rename(columns={'size': 'excluded_origin_model_groups'}))
    exclusions['excluded_operation_decisions'] = exclusions['excluded_origin_model_groups'] * 2
models = ['persistence_last', 'ridge_trm', 'arima_010', 'arima_011', 'arima_010_drift']
complete = {}
for (partition, model), group in forecasts.groupby(['partition', 'model']):
    group_issues = validate_forecast_grid(group, horizons, strict=False)
    bad = set(group_issues['forecast_origin']) if not group_issues.empty else set()
    complete[(partition, model)] = {origin for origin in group['forecast_origin'].unique() if origin not in bad and group[group['forecast_origin'] == origin]['horizon'].nunique() == len(horizons)}
common = {partition: set.intersection(*[complete.get((partition, model), set()) for model in models]) for partition in ['validation', 'test']}
comparable = forecasts[forecasts.apply(lambda row: row['forecast_origin'] in common.get(row['partition'], set()), axis=1)].copy()
exclusions

In [ ]:
decisions, summary = evaluate_business_value(comparable, usd_amount=usd_amount, horizons=horizons, strict=True)
later = (decisions.assign(later=decisions['selected_horizon'] > 1)
    .groupby(['partition', 'model', 'operation'], as_index=False)['later'].mean())
later['later_than_1_pct'] = later['later'] * 100
business_summary = summary.merge(later.drop(columns='later'), on=['partition', 'model', 'operation'])
display(business_summary.sort_values(['partition', 'operation', 'model']))
horizon_distribution = (decisions.groupby(['partition', 'model', 'operation', 'selected_horizon']).size().rename('decisions').reset_index())
horizon_distribution['pct_of_decisions'] = horizon_distribution['decisions'] / horizon_distribution.groupby(['partition', 'model', 'operation'])['decisions'].transform('sum') * 100
display(horizon_distribution)
display(decisions[['partition', 'model', 'operation', 'forecast_origin', 'selected_horizon', 'predicted_trm_at_selection', 'realized_trm_at_selection', 'realized_trm_horizon_1', 'gross_cop_benefit']].head(20))

## Limitations

The primary experiment uses only TRM-derived features. External predictors are not silently substituted because economic-period dates do not establish when observations were published or available to a historical forecaster. Gross benefits are hypothetical comparisons before spreads, fees, taxes, liquidity, and execution constraints.